## 8. Handling Missing Values

### How NumPy represents missing data
| Representation | Use case |
|---------------|----------|
| `np.nan` | Float arrays (IEEE 754 standard) |
| `np.inf / -np.inf` | Overflow / division |
| `np.ma.masked` | Any dtype - integer missing values |

### NaN is contagious — use nan-safe functions
| Regular | NaN-safe |
|---------|----------|
| `np.sum` | `np.nansum` |
| `np.mean` | `np.nanmean` |
| `np.std` | `np.nanstd` |
| `np.min/max` | `np.nanmin/nanmax` |
| `np.median` | `np.nanmedian` |
| `np.argmin/max` | `np.nanargmin/nanargmax` |

### Detection
```python
np.isnan(arr)      # boolean mask
np.isinf(arr)      # inf positions
np.isfinite(arr)   # not NaN and not Inf
```

In [9]:
import numpy as np

In [11]:
# NaN basics
a = np.array([1., np.nan, 3., np.nan, 5.])
print('Array    :', a)
# np.isnan()
print('isnan    :', np.isnan(a))
# count of nan
print('NaN count:', np.isnan(a).sum())

Array    : [ 1. nan  3. nan  5.]
isnan    : [False  True False  True False]
NaN count: 2


In [12]:
# np.isnan(array)

arr = np.array([1,2,np.nan,4,np.nan,6])
print(np.isnan(arr))
#not
print(np.nan == np.nan)

[False False  True False  True False]
False


In [13]:
# np.nan_to_num(array, nan=value) default = 0

arr = np.array([1,2,np.nan,4,np.nan,6])
print('Array    :', arr)

cleaned_arr = np.nan_to_num(arr)
print(cleaned_arr)

cleaned_arr = np.nan_to_num(arr, nan=10)
print(cleaned_arr)

Array    : [ 1.  2. nan  4. nan  6.]
[1. 2. 0. 4. 0. 6.]
[ 1.  2. 10.  4. 10.  6.]


In [8]:
# np.isinf() 
# ex: 10^10000, 1/0

arr = np.array([1,2,np.inf,4,-np.inf,6])
print(np.isinf(arr))

cleaned_arr = np.nan_to_num(arr, posinf=1000, neginf=-1000)
print(cleaned_arr)

[False False  True False  True False]
[    1.     2.  1000.     4. -1000.     6.]


In [14]:
# NaN contagion

a = np.array([1., np.nan, 3., np.nan, 5.])
print('Array    :', a)

print('sum→', np.sum(a), '  nansum→', np.nansum(a))
print('mean→', np.mean(a), '  nanmean→', np.nanmean(a))

Array    : [ 1. nan  3. nan  5.]
sum→ nan   nansum→ 9.0
mean→ nan   nanmean→ 3.0


In [17]:
# valid
a = np.array([1., np.nan, 3., np.nan, 5.])
print('Array    :', a)

valid = a[~np.isnan(a)]
print('Valid values:', valid)
print('NaN indices :', np.where(np.isnan(a))[0])

Array    : [ 1. nan  3. nan  5.]
Valid values: [1. 3. 5.]
NaN indices : [1 3]


In [18]:
# Imputation strategies
data = np.array([10., np.nan, 30., np.nan, 50., np.nan, 70.])
print('Original   :', data)

# 1. Mean imputation
d_mean = data.copy()
d_mean[np.isnan(d_mean)] = np.nanmean(d_mean)
print('Fill mean  :', d_mean)

# 2. Median imputation
d_med = data.copy()
d_med[np.isnan(d_med)] = np.nanmedian(d_med)
print('Fill median:', d_med)

# 3. Forward fill
d_ff = data.copy()
for i in range(1, len(d_ff)):
    if np.isnan(d_ff[i]): d_ff[i] = d_ff[i-1]
print('Fwd fill   :', d_ff)

# 4. Linear interpolation
nan_pos  = np.isnan(data)
x        = np.arange(len(data))
d_interp = data.copy()
d_interp[nan_pos] = np.interp(x[nan_pos], x[~nan_pos], data[~nan_pos])
print('Interp     :', d_interp)

Original   : [10. nan 30. nan 50. nan 70.]
Fill mean  : [10. 40. 30. 40. 50. 40. 70.]
Fill median: [10. 40. 30. 40. 50. 40. 70.]
Fwd fill   : [10. 10. 30. 30. 50. 50. 70.]
Interp     : [10. 20. 30. 40. 50. 60. 70.]


In [19]:
# 2-D missing value report
rng = np.random.default_rng(7)
mat = rng.random((5,4)).round(2)
mat[rng.random((5,4))<0.25] = np.nan
print('Matrix with NaN:\n', mat)
print('NaN per column:', np.isnan(mat).sum(axis=0))
print('NaN per row   :', np.isnan(mat).sum(axis=1))
print('Col means     :', np.nanmean(mat,axis=0).round(2))

Matrix with NaN:
 [[ nan  nan 0.78  nan]
 [ nan 0.87 0.01 0.82]
 [0.8  0.47 0.3   nan]
 [ nan  nan 0.5   nan]
 [1.    nan 0.62  nan]]
NaN per column: [3 3 0 4]
NaN per row   : [3 1 1 3 2]
Col means     : [0.9  0.67 0.44 0.82]
